# CryptoQuant BTC 데이터 추출기 (Google Colab용)

CryptoQuant에서 BTC 가격과 미결제 약정(Open Interest) 데이터를 추출합니다.

## 추출 데이터
- 📅 **날짜** (Date)
- 💰 **BTC 가격** (Price USD)
- 📊 **미결제 약정** (Open Interest)

## 1. 필요한 패키지 설치

In [ ]:
# Google Colab용 패키지 설치
!pip install -q google-colab-selenium pandas beautifulsoup4

print("✅ 설치 완료!")

## 2. 스크래퍼 코드 로드

In [ ]:
# GitHub에서 스크래퍼 다운로드
!wget -q https://raw.githubusercontent.com/ym0179/coin_monitor/claude/fix-highcharts-tooltip-01H4VypGM6KYkxLkzdAgoasy/cryptoquant_scraper.py

print("✅ 스크래퍼 다운로드 완료!")

## 3. 라이브러리 임포트

In [ ]:
from cryptoquant_scraper import CryptoQuantScraper
import pandas as pd
from google.colab import files

print("✅ 라이브러리 임포트 완료!")

## 4. 데이터 추출

In [ ]:
# CryptoQuant URL
url = "https://cryptoquant.com/asset/btc/chart/derivatives/open-interest?exchange=all_exchange&symbol=all_symbol&window=DAY&sma=0&ema=0&priceScale=log&metricScale=linear&chartStyle=line"

print("="*80)
print("CryptoQuant BTC 데이터 추출 시작")
print("="*80)

# 스크래퍼 초기화 (Colab에서는 headless 자동 처리됨)
scraper = CryptoQuantScraper(headless=True)

try:
    # 데이터 추출
    print("\n데이터 추출 중... (약 10-15초 소요)")
    df = scraper.extract_highcharts_data(url)
    
    if df is not None and not df.empty:
        print("\n" + "="*80)
        print("✅ 데이터 추출 성공!")
        print("="*80)
        print(f"\n총 데이터 수: {len(df):,}개")
        print(f"컬럼: {', '.join(df.columns)}")
        
        # 최신 데이터 표시
        if not df.empty:
            print("\n" + "-"*80)
            print("📊 최신 데이터")
            print("-"*80)
            
            latest = df.iloc[-1]
            print(f"📅 날짜: {latest['date']}")
            
            if 'price_usd' in df.columns:
                print(f"💰 BTC 가격: ${latest['price_usd']:,.2f}")
            
            if 'open_interest' in df.columns:
                print(f"📊 미결제 약정: ${latest['open_interest']:,.2f}")
    else:
        print("\n❌ 데이터를 추출하지 못했습니다.")
        
finally:
    scraper.close()
    print("\n" + "="*80)
    print("프로그램 종료")
    print("="*80)

## 5. 데이터 미리보기

In [ ]:
if df is not None and not df.empty:
    print("처음 10개 데이터:")
    display(df.head(10))
    
    print("\n마지막 10개 데이터:")
    display(df.tail(10))
    
    print("\n기본 통계:")
    display(df.describe())

## 6. CSV 파일로 저장 및 다운로드

In [ ]:
if df is not None and not df.empty:
    # CSV 파일로 저장
    filename = 'cryptoquant_btc_data.csv'
    df.to_csv(filename, index=False)
    print(f"✅ 데이터가 '{filename}' 파일로 저장되었습니다.")
    
    # 파일 다운로드
    print("\n파일 다운로드 중...")
    files.download(filename)
    print("✅ 다운로드 완료!")

## 7. 데이터 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if df is not None and not df.empty:
    # 날짜별 정렬
    df_sorted = df.sort_values('date')
    
    # 그래프 생성
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
    
    # BTC 가격 그래프
    if 'price_usd' in df_sorted.columns:
        ax1.plot(df_sorted['date'], df_sorted['price_usd'], 
                color='#4C32EA', linewidth=2, label='BTC Price')
        ax1.set_title('💰 BTC Price (USD)', fontsize=16, fontweight='bold', pad=20)
        ax1.set_ylabel('Price (USD)', fontsize=12)
        ax1.grid(True, alpha=0.3)
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        ax1.legend()
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 미결제 약정 그래프
    if 'open_interest' in df_sorted.columns:
        ax2.plot(df_sorted['date'], df_sorted['open_interest'], 
                color='#7F3FBF', linewidth=2, label='Open Interest')
        ax2.set_title('📊 Open Interest', fontsize=16, fontweight='bold', pad=20)
        ax2.set_ylabel('Open Interest (USD)', fontsize=12)
        ax2.set_xlabel('Date', fontsize=12)
        ax2.grid(True, alpha=0.3)
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        ax2.legend()
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("✅ 그래프 생성 완료!")

## 8. 특정 기간 데이터 필터링

In [ ]:
if df is not None and not df.empty:
    # 최근 30일 데이터만
    df_recent = df.tail(30)
    print(f"최근 30일 데이터: {len(df_recent)}개")
    display(df_recent)
    
    # 특정 날짜 이후 데이터 (예시)
    # df_2025 = df[df['date'] >= '2025-01-01']
    # print(f"\n2025년 이후 데이터: {len(df_2025)}개")

## 9. 데이터 분석

In [ ]:
if df is not None and not df.empty:
    print("="*80)
    print("📈 데이터 분석")
    print("="*80)
    
    if 'price_usd' in df.columns:
        print("\n💰 BTC 가격:")
        print(f"  최고가: ${df['price_usd'].max():,.2f}")
        print(f"  최저가: ${df['price_usd'].min():,.2f}")
        print(f"  평균가: ${df['price_usd'].mean():,.2f}")
        print(f"  중간값: ${df['price_usd'].median():,.2f}")
    
    if 'open_interest' in df.columns:
        print("\n📊 미결제 약정:")
        print(f"  최대: ${df['open_interest'].max():,.2f}")
        print(f"  최소: ${df['open_interest'].min():,.2f}")
        print(f"  평균: ${df['open_interest'].mean():,.2f}")
        print(f"  중간값: ${df['open_interest'].median():,.2f}")
    
    print("\n📅 기간:")
    print(f"  시작일: {df['date'].min()}")
    print(f"  종료일: {df['date'].max()}")
    print(f"  총 기간: {(df['date'].max() - df['date'].min()).days}일")

---

## 사용 팁

1. **에러 발생 시**: 셀을 다시 실행해보세요. 네트워크 문제일 수 있습니다.
2. **데이터가 없을 때**: 웹사이트가 변경되었을 수 있습니다. GitHub 이슈를 확인하세요.
3. **더 많은 데이터**: URL의 `window=DAY`를 `window=HOUR`로 변경하면 시간별 데이터를 얻을 수 있습니다.

## 문제 해결

- **Chrome 오류**: 첫 번째 셀(패키지 설치)을 다시 실행하세요.
- **타임아웃**: `time.sleep` 값을 늘려보세요.
- **데이터 형식**: 추출된 데이터를 확인하고 필요에 맞게 가공하세요.

---

**GitHub**: [coin_monitor](https://github.com/ym0179/coin_monitor)
